# KíLín — Code-Project Evaluation Notebook

**Routing, repository understanding, code relationships, conversational
continuity, tool execution, and evidence quality**

This standalone notebook tests KíLín against a fresh **development
snapshot of KíLín itself**, importing only `kilin/` and `tutorials/`.
Its objective is to evaluate both the notebook's recording/reconciliation
and KíLín's actual code-oriented behavior. No development source files
are changed or executed by the evaluation.

The project saves and indexes `.py`, `.ipynb`, `.m`, `.md`, `.json`, `.js`,
and `.html` files under the existing code-project type. The extension list
is applied to this **experiment's saved profile**; the package defaults
and format extractors remain unchanged. All sources follow KíLín's normal
file-selection, reading, and indexing rules.

### What is tested

1. **Routing:** selected intent, tool, retrieval policy, and prior-turn links.
2. **Scope:** which precise source paths occur in saved retrieved evidence,
   independently of how many files a tool scanned.
3. **Repository facts:** definitions, line numbers, imports and some
   syntactic relationships computed independently with Python's AST.
4. **Answer quality:** limited keyword checks and selected source-grounded
   reference facts; qualitative interpretation requires manual review.
5. **Execution:** completion, citations, evidence provenance, trace integrity,
   model usage, and optional tool/answering/router comparisons.

A routing `MATCH` does **not** mean an answer is correct. The independent
source checks are also not semantic correctness judges. Routing-only probes
do not execute tools or commit conversational answers.

### Recorded summaries and displays

Compact rows show KíLín's own `conversation.turn.representation` summary
from the native trace. A saved checkpoint representation is used only if
the event is unavailable; any disagreement is flagged. No separate summary
is reconstructed from the raw answer. Expand **Answer and routing details**
to inspect the unmodified response (including thinking), exact source
paths, prior turns, and tool diagnostics.

`---` means unscored or inapplicable, **not** success. Scientific/code
accuracy is assessed independently of routing and keyword anchors.

### Execution

Run sections in order from a kernel started inside the KíLín checkout.
Each run creates a new workspace under `sandbox/`. Expensive scans, complete
file reviews, and model comparisons are controlled by flags in Section 1.
Tools never obtain network permission from this notebook. Code-generation
answers are **text only**; no generated code is automatically executed.

The notebook records source hashes and Git information for every imported
file. It makes no release-version assumption: the current checkout is the
benchmark source. Traces can contain source code and model reasoning; keep
exported workspaces private when appropriate.


## 1. Experiment configuration

Run from the current KíLín development checkout. This experiment copies
source folders into its own independent workspace and records the Git HEAD
and working-tree status where available. Source hashes are authoritative
even when the checkout is dirty or not managed by Git.

By default, core scenarios and lightweight probes run; expensive tools,
exhaustive analysis and model comparisons are opt-in. Adjust flags before
executing the setup cell. All model IDs must already be installed locally
when `ALLOW_DOWNLOAD=False`.


In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess

EXPERIMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PROJECT_NAME = f"trial_code_routing_{EXPERIMENT_ID}"
ALLOW_DOWNLOAD = False

MODEL_IDS = {
    "qwen8b": "Qwen/Qwen3-8B-MLX-4bit",
    "qwen14b": "Qwen/Qwen3-14B-MLX-4bit",
    "qwen38_27b": "lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly",
}
PRIMARY_MODEL = "qwen8b"
ROUTER_MODEL = "Qwen/Qwen3-4B-MLX-4bit"
COMPARISON_MODELS = ("qwen8b", "qwen38_27b")
COMPARISON_ROUTERS = (
    "Qwen/Qwen3-4B-MLX-4bit",
    "Qwen/Qwen3-8B-MLX-4bit",
)

CODE_EXTENSIONS = (
    ".py", ".ipynb", ".m", ".md", ".json", ".js", ".html"
)
RETRIEVAL_COUNT = 10
RUN_LONG_TOOLS = False
RUN_FULL_SOURCE_READ = False
RUN_CODE_REVIEW = False
RUN_MODEL_COMPARISON = False
RUN_ROUTER_COMPARISON = False
RUN_GUI_EXPORT_TEST = False

assert PRIMARY_MODEL in MODEL_IDS
assert all(key in MODEL_IDS for key in COMPARISON_MODELS)

repo_root = Path.cwd().resolve()
while not (
    (repo_root / "pyproject.toml").is_file()
    and (repo_root / "kilin/project_types/code.json").is_file()
    and (repo_root / "tutorials").is_dir()
):
    if repo_root == repo_root.parent:
        raise RuntimeError("Run inside the KíLín development repository.")
    repo_root = repo_root.parent
project_dir = repo_root / "sandbox" / PROJECT_NAME


def _git_info(arguments):
    """Read optional source-version details without changing the checkout.

    Parameters
    ----------
    arguments : list of str
        Arguments passed to the Git executable.

    Returns
    -------
    str or None
        Standard output, or None when Git cannot provide a result.

    """
    try:
        result = subprocess.run(
            ["git", "-C", str(repo_root), *arguments],
            capture_output=True, text=True, timeout=15, check=False,
        )
    except (OSError, subprocess.TimeoutExpired):
        return None
    return result.stdout.strip() if result.returncode == 0 else None


SOURCE_VERSION = {
    "git_commit": _git_info(["rev-parse", "HEAD"]),
    "git_status": _git_info([
        "status", "--porcelain", "--untracked-files=normal",
        "--", "kilin", "tutorials",
    ]),
}
print("Development repository:", repo_root)
print("Experiment:", PROJECT_NAME)
print("Destination:", project_dir)
print("Git commit:", SOURCE_VERSION["git_commit"] or "unavailable")
print(
    "Source tree modified:",
    "unavailable" if SOURCE_VERSION["git_status"] is None
    else bool(SOURCE_VERSION["git_status"]),
)
print("Extensions:", ", ".join(CODE_EXTENSIONS))
print("Answering model:", MODEL_IDS[PRIMARY_MODEL])
print("Router:", ROUTER_MODEL)


## 2. Recording and evaluation helpers

The helpers use `RAGSession` and KíLín's native router. They record actual
answers, selected tools, previous-turn links, model calls, and trace-authored
turn representations. They do not change KíLín's retrieval or generation
logic. Explicit expected **source paths** disambiguate same-named files
such as `__init__.py`.

A routing decision is scored separately from source scope, execution,
trace consistency and content anchors. Full answers remain available
unchanged in the expanded details. Code correctness requires checking
actual definitions and relationships; there is no model-as-judge loop.


In [ ]:
import ast
import csv
import re
import html
import json
from contextlib import contextmanager
from time import perf_counter

from IPython.display import HTML, display

import kilin

ACTIVE_PACKAGE_ROOT = Path(kilin.__file__).resolve().parents[1]
if ACTIVE_PACKAGE_ROOT != repo_root:
    raise RuntimeError(
        "The imported kilin package is not the development "
        "checkout. Start the kernel from this checkout. "
        f"Loaded {ACTIVE_PACKAGE_ROOT}; expected {repo_root}."
    )

from kilin.answering import route_question, strip_thinking
from kilin.conversations import ConversationStore
from kilin.project_type import project_type
from kilin.rag import RAGSession

CASE_RESULTS = []
CASE_INDEX = {}
CONVERSATION_INDEX = {}


def code_intents():
    """Return configured code intents without hard-coded tool names.

    Returns
    -------
    dict
        Current name-to-configuration mapping from code.json.

    """
    path = repo_root / "kilin" / "project_types" / "code.json"
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)["intents"]


def expected_tool(intent):
    """Return the tool configured for an intent, or None for model answers.

    Parameters
    ----------
    intent : str or None
        Intent identifier in the current code project type.

    Returns
    -------
    str or None
        Configured tool name, if present.

    """
    tool = code_intents().get(intent, {}).get("tool") or {}
    return tool.get("name")


def _source_names(evidence):
    """Identify filenames in saved source evidence, retaining order.

    Parameters
    ----------
    evidence : sequence
        Evidence objects persisted by RAGSession.

    Returns
    -------
    list of str
        Unique document basenames.

    """
    found = []
    for item in evidence:
        path = getattr(item, "path", None)
        if path is None and isinstance(item, dict):
            path = item.get("path") or item.get("source")
        if path:
            name = Path(str(path)).name
            if name not in found:
                found.append(name)
    return found


def _trace_operation(stages, stage_name, suffix):
    """Locate an operation by semantic suffix, not sequence number.

    Parameters
    ----------
    stages : dict
        One native trace turn's stages.
    stage_name : str
        Stage name such as ``01_question``.
    suffix : str
        End of the operation name, without its sequence prefix.

    Returns
    -------
    dict
        Matching operation or an empty mapping.

    """
    operations = stages.get(stage_name, {}).get("operations", {})
    return next(
        (value for key, value in operations.items() if key.endswith(suffix)),
        {},
    )


def _trace_turns(path):
    """Load committed turns from the native conversation trace.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace JSON path.

    Returns
    -------
    dict
        Native turns keyed by their recorded one-based IDs.

    """
    try:
        with path.open(encoding="utf-8") as stream:
            trace = json.load(stream)
        turns = trace["sessions"]["05_conversations"]["turns"]
        return turns
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return {}


def _recorded_representation(turn, traced):
    """Read the KíLín-created summary, preferring its trace event.

    Parameters
    ----------
    turn : ConversationTurn
        Committed turn containing KíLín's saved representation.
    traced : dict or None
        Representation from ``conversation.turn.representation``.

    Returns
    -------
    dict
        Recorded summary, provenance, and trace/checkpoint agreement.

    Notes
    -----
    Neither source is reconstructed from the raw answer. If both recorded
    representations exist, they should agree on their semantic fields.

    """
    traced = traced if isinstance(traced, dict) else {}
    parameters = getattr(turn, "parameters", {}) or {}
    checkpoint = parameters.get("representation") or {}
    if not isinstance(checkpoint, dict):
        checkpoint = {}
    trace_ok = isinstance(traced.get("summary"), str)
    saved_ok = isinstance(checkpoint.get("summary"), str)
    if trace_ok:
        representation = traced
        origin = "trace"
    elif saved_ok:
        representation = checkpoint
        origin = "checkpoint"
    else:
        representation = {}
        origin = "unavailable"
    consistent = None
    if trace_ok and saved_ok:
        fields = ("question", "summary", "sources")
        consistent = all(
            traced.get(field) == checkpoint.get(field)
            for field in fields
        )
    return {
        "summary": representation.get("summary", ""),
        "sources": list(representation.get("sources") or []),
        "origin": origin,
        "consistent": consistent,
    }


def _native_route(path, turn_number):
    """Read routing and timing details from one saved native trace turn.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace path.
    turn_number : int
        One-based turn number in this conversation.

    Returns
    -------
    dict
        Router, history, retrieval, and model-call diagnostics.

    """
    turns = _trace_turns(path)
    recorded = turns.get(f"turn_{turn_number:04d}", {})
    if not recorded:
        return {"trace_available": False}
    stages = recorded.get("stages", {})

    routed = _trace_operation(
        stages, "01_question", "intent_classification_decided"
    ).get("details", {})
    history = _trace_operation(
        stages, "01_question", "history_selection_result"
    ).get("details", {})
    retrieval = _trace_operation(
        stages, "03_retrieval", "retrieval_result"
    ).get("details", {})
    settings = _trace_operation(
        stages, "01_question", "turn_settings_snapshot"
    ).get("details", {})
    representation = _trace_operation(
        stages, "01_question", "conversation_turn_representation"
    ).get("details", {})
    model_calls = []
    for stage in stages.values():
        for name, value in stage.get("operations", {}).items():
            if name.endswith("llm_call_output"):
                model_calls.append(value.get("details", {}))
    return {
        "trace_available": True,
        "classification": routed.get("classification", {}),
        "shortcut": routed.get("shortcut"),
        "fallback": routed.get("fallback"),
        "dropped_turns": routed.get("dropped_turns", []),
        "router_raw": routed.get("raw"),
        "history": history,
        "retrieval": retrieval,
        "settings": settings,
        "model_calls": model_calls,
        "representation": representation,
    }

def _source_paths(evidence):
    """Return distinct saved source paths relative to the source root.

    Parameters
    ----------
    evidence : sequence
        Saved evidence objects or their checkpoint dictionary forms.

    Returns
    -------
    list of str
        Normalized paths, with distinct directories preserved.

    """
    found = []
    for item in evidence:
        path = getattr(item, "path", None)
        if path is None and isinstance(item, dict):
            path = item.get("path") or item.get("source")
        if not path:
            continue
        value = str(path).replace("\\", "/")
        if "/sources/" in value:
            value = value.split("/sources/", 1)[1]
        elif value.startswith("sources/"):
            value = value[len("sources/"):]
        if value not in found:
            found.append(value)
    return found


def _has_source(paths, target):
    """Match an expected repository-relative path without basename loss.

    Parameters
    ----------
    paths : sequence of str
        Saved source paths.
    target : str
        Expected path, such as ``kilin/rag.py``.

    Returns
    -------
    bool
        Whether the saved evidence includes that source path.

    """
    value = target.replace("\\", "/").strip("/")
    return any(
        path == value or path.endswith("/" + value)
        for path in paths
    )


In [ ]:
def _gui_callbacks(events, choice="default", confirmation="quick"):
    """Create non-networked callbacks and record each scripted decision.

    Parameters
    ----------
    events : list
        Mutable list populated with GUI-equivalent interactions.
    choice : str or tuple, optional
        ``default``, ``first20``, ``save_all``, ``cancel``, or a tuple
        ``("document", filename)``. A value of ``none`` omits the adapter.
    confirmation : str, optional
        ``quick``, ``continue``, or ``cancel`` for model plans. Network plans
        are always cancelled.

    Returns
    -------
    tuple
        Choice callback and confirmation callback for ``RAGSession.ask``.

    """

    def choose(question, options):
        selected = None
        if isinstance(choice, tuple) and choice[0] == "document":
            target = choice[1].casefold()
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if (
                        item.split(" (", 1)[0].casefold() == target
                        or item.casefold().endswith("/" + target)
                    )
                ),
                None,
            )
        elif choice in ("default", "first20"):
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item == "Show the first 20 here"
                ),
                None,
            )
            if selected is None and choice == "default" and options:
                if "which paper" in question.casefold() or (
                    "which document" in question.casefold()
                ):
                    selected = 0
        elif choice == "save_all":
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item.startswith("Save all ")
                    and item.endswith(" to outputs/ as Markdown")
                ),
                None,
            )
        events.append(
            {
                "kind": "choice",
                "question": question,
                "options": list(options),
                "selected": selected,
            }
        )
        return selected

    def confirm(plan):
        decision = (
            "cancel"
            if getattr(plan, "operation", "model") == "network"
            else confirmation
        )
        events.append(
            {
                "kind": "confirmation",
                "tool": plan.tool,
                "operation": getattr(plan, "operation", "model"),
                "summary": plan.summary,
                "items": plan.items,
                "batches": plan.batches_to_run,
                "quick_items": plan.quick_items,
                "selected": decision,
            }
        )
        return decision

    choice_callback = None if choice == "none" else choose
    return choice_callback, confirm




CONVERSATION_SECTIONS = {
    "empty-project": "3.1",
    "direct-routes": "5",
    "code-relations": "6.1",
    "code-continuity": "6.2",
    "results-tools": "7",
    "gui-paths": "7.1",
    "routing-boundaries": "8",
    "hard-scope": "9",
    "code-ground-truth": "9.1",
    "optional-review": "10",
    "optional-file-review": "10",
    "code-generation": "11",
    "answer-models-seed": "12",
    "router-seed": "13",
}


def _conversation_name(label):
    """Generate a persistent name using the section and recorded cases.

    Parameters
    ----------
    label : str
        Logical test group identifier.

    Returns
    -------
    str
        Notebook section and first/last recorded case IDs.

    """
    ids = [
        record["id"] for record in CASE_RESULTS
        if record["label"] == label
    ]
    if not ids:
        return label
    section = CONVERSATION_SECTIONS.get(label)
    if label.startswith("answer-model-"):
        section = "12"
    elif label.startswith("router-model-"):
        section = "13"
    span = ids[0] if len(ids) == 1 else f"{ids[0]}-{ids[-1]}"
    return f"{section}. {span}" if section else f"{label}: {span}"


@contextmanager
def conversation(label, *, model=None, router=None, branch_from=None):
    """Open an independent test conversation and close it reliably.

    Parameters
    ----------
    label : str
        Human-readable scenario group name.
    model : str or None, optional
        Key in MODEL_IDS; defaults to PRIMARY_MODEL.
    router : str or None, optional
        Router model identifier; defaults to ROUTER_MODEL.
    branch_from : str or None, optional
        Existing conversation ID whose saved history is cloned into this
        conversation. A new independent conversation is the default.

    Yields
    ------
    dict
        Active session, trace path, and case-to-turn registry.

    """
    selected = model or PRIMARY_MODEL
    if selected not in MODEL_IDS:
        raise ValueError(f"Unknown model key: {selected}")
    if label in CONVERSATION_INDEX:
        raise ValueError(f"Conversation label already used: {label}")

    store = ConversationStore(project_dir)
    if branch_from is not None:
        store.ensure()
    info = store.create(branch_from=branch_from)
    path = store.state_path(info)
    session = None
    try:
        session = RAGSession(
            project_dir,
            conversation_path=path,
            router_model=router or ROUTER_MODEL,
        )
        session.use_chat_model(MODEL_IDS[selected])
        context = {
            "label": label,
            "store": store,
            "info": info,
            "path": path,
            "session": session,
            "case_turn_ids": {},
            "model": selected,
            "router": router or ROUTER_MODEL,
            "parent": branch_from,
        }
        CONVERSATION_INDEX[label] = {
            "path": str(path.relative_to(project_dir)),
            "model": selected,
            "router": router or ROUTER_MODEL,
            "parent": branch_from,
        }
        print(f"Conversation {label}: {path.name}")
        print("  Answering:", session.chat_session.model_id)
        print("  Router:", session.router_model)
        yield context
    finally:
        if session is not None:
            session.close()
            name = _conversation_name(label)
            renamed = store.rename(info.conversation_id, name)
            CONVERSATION_INDEX[label]["name"] = renamed.name


def _expected_links(context, case_ids):
    """Resolve named earlier cases to one-based conversation turn IDs.

    Parameters
    ----------
    context : dict
        Active conversation context.
    case_ids : tuple
        Previously run case identifiers.

    Returns
    -------
    list of int
        One-based expected turn numbers.

    """
    missing = [
        case_id
        for case_id in case_ids
        if case_id not in context["case_turn_ids"]
    ]
    if missing:
        raise ValueError(f"Missing earlier cases: {missing}")
    return [context["case_turn_ids"][item] for item in case_ids]


def _check_generated_function(record):
    """Check generated Python without executing it.

    Parameters
    ----------
    record : dict
        Saved answer and expected top-level function identifier.

    Returns
    -------
    list of str
        Syntax and formatting problems for manual assessment.

    """
    expected = record.get("expected_generated_function")
    if not expected:
        return []
    answer = strip_thinking(record.get("answer") or "")
    match = re.search(
        r"```(?:python)?\s*\n([\s\S]*?)```", answer,
        flags=re.IGNORECASE,
    )
    if match is None:
        return ["no fenced Python code to validate"]
    source = match.group(1)
    try:
        tree = ast.parse(source)
    except SyntaxError as error:
        return [f"generated Python syntax: {error.msg}"]
    problems = []
    if not any(
        isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef))
        and item.name == expected for item in tree.body
    ):
        problems.append(f"missing generated function: {expected}")
    if any(len(line) > 79 for line in source.splitlines()):
        problems.append("generated Python exceeds 79 columns")
    return problems


def _check_record(record):
    """Evaluate routing, source scope, and diagnostics independently.

    Parameters
    ----------
    record : dict
        Completed routing probe or committed conversation turn.

    Returns
    -------
    dict
        Separate lists of routing, scope, execution, trace, and content
        concerns. Content anchors are not code correctness proofs.

    """
    checks = {
        "routing": [],
        "scope": [],
        "execution": [],
        "trace": [],
        "content": [],
    }
    route = checks["routing"]
    expected = record.get("expected_intent")
    if expected is not None and record.get("intent") != expected:
        route.append("intent")
    if record["mode"] == "full" and expected is not None:
        if record.get("tool") != record.get("expected_tool"):
            route.append("tool")
    standalone = record.get("expected_standalone")
    if standalone is not None and record.get("standalone") != standalone:
        route.append("standalone")
    actual = set(record.get("related_turn_ids", []))
    required = set(record.get("expected_related_ids", []))
    forbidden = set(record.get("forbidden_related_ids", []))
    if not required.issubset(actual):
        route.append("missing prior-turn link")
    if forbidden & actual:
        route.append("unrelated prior-turn link")
    if record.get("expected_retrieval") is not None:
        if record.get("retrieval_mode") != record["expected_retrieval"]:
            route.append("retrieval policy")
    if record.get("intent") in record.get("disallowed_intents", []):
        route.append("disallowed intent")

    if record["mode"] == "full":
        sources = set(record.get("sources", []))
        if not set(record.get("expected_sources", [])).issubset(sources):
            checks["scope"].append("missing source in retrieved evidence")
        if set(record.get("forbidden_sources", [])) & sources:
            checks["scope"].append("out-of-scope retrieved source")
        paths = record.get("source_paths", [])
        for name in record.get("expected_paths", []):
            if not _has_source(paths, name):
                checks["scope"].append(f"missing source path: {name}")
        for name in record.get("forbidden_paths", []):
            if _has_source(paths, name):
                checks["scope"].append(f"unexpected source: {name}")
        checks["content"].extend(_check_generated_function(record))
        if record.get("invalid_citations"):
            checks["execution"].append("citation warning (check quotes)")
        if record.get("completion"):
            checks["execution"].append("incomplete answer")
        if not record.get("trace", {}).get("trace_available"):
            checks["trace"].append("native trace unavailable")
        if record.get("summary_source") != "trace":
            checks["trace"].append("summary not from trace event")
        if record.get("summary_consistent") is False:
            checks["trace"].append("trace/checkpoint summary mismatch")
        logged_model = record.get("trace", {}).get(
            "settings", {}
        ).get("chat_model")
        if logged_model and logged_model != record.get("model"):
            checks["trace"].append("active-model trace mismatch")
        for prior in record.get("previous", []):
            if prior.get("summary_consistent") is False:
                checks["trace"].append("prior summary mismatch")
                break
        answer = strip_thinking(record.get("answer") or "").casefold()
        for term in record.get("expected_answer_terms", []):
            if term.casefold() not in answer:
                checks["content"].append(f"missing answer anchor: {term}")
    return checks


def _grade_record(record):
    """Attach independent diagnostic fields and a routing-only verdict.

    Parameters
    ----------
    record : dict
        Populated case record, updated in place.

    """
    checks = _check_record(record)
    record["checks"] = checks
    record["issues"] = [
        f"{kind}: {message}"
        for kind, problems in checks.items()
        for message in problems
    ]
    record["status"] = (
        "REVIEW"
        if checks["routing"]
        else "MATCH" if record.get("expected_intent") else "---"
    )
    record["scope_status"] = (
        "REVIEW" if checks["scope"]
        else "OK" if record.get("expected_sources")
        or record.get("forbidden_sources")
        or record.get("expected_paths")
        or record.get("forbidden_paths") else "---"
    )


In [ ]:
def _brief(value, limit=95):
    """Return short escaped text for a compact diagnostic table.

    Parameters
    ----------
    value : object
        Value to show.
    limit : int, optional
        Maximum characters before truncation.

    Returns
    -------
    str
        Escaped, whitespace-normalized label.

    """
    text = " ".join(str(value if value is not None else "-").split())
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return html.escape(text)


def _final_answer_text(answer):
    """Remove reasoning for content checks, not answer summaries.

    Parameters
    ----------
    answer : str or None
        Raw answer, potentially containing model reasoning.

    Returns
    -------
    str
        Answer text excluding thinking blocks.

    """
    return strip_thinking(answer or "")


def show_case(record):
    """Show one compact table and optional full answer diagnostics.

    Parameters
    ----------
    record : dict
        Recorded routing-only or end-to-end test case.

    """
    expected = record.get("expected_intent") or "---"
    actual = record.get("intent") or "-"
    tool = record.get("tool") or "---"
    if record["mode"] == "route-only":
        tool = "---"
    else:
        wanted = record.get("expected_tool") or "---"
        tool = f"{wanted} → {tool}"
    related = record.get("related_turn_ids", [])
    need = record.get("expected_related_ids", [])
    tokens = record.get("usage", {})
    token_text = (
        f"{tokens.get('input_tokens', 0)}/" f"{tokens.get('output_tokens', 0)}"
        if tokens
        else "-"
    )
    source_names = record.get("sources", [])
    source_text = f"{len(source_names)}: " + ", ".join(source_names)
    scanned = record.get("files_scanned")
    if scanned is not None:
        source_text += f" ({scanned} scanned)"
    status = record["status"]
    checks = record.get("checks", {})
    other = ", ".join(
        name for name in ("scope", "execution", "trace", "content")
        if checks.get(name)
    ) or "---"
    body = "".join(
        "<td style='padding:4px 7px;border-bottom:1px solid #bbb'>"
        + value
        + "</td>"
        for value in (
            _brief(f"{expected} → {actual}", 55),
            _brief(tool, 22),
            _brief(f"need {need}; got {related}", 29),
            _brief(record.get("retrieval_mode") or "---", 12),
            _brief(source_text, 47),
            _brief(token_text, 22),
            _brief(f"{record.get('seconds', 0):.2f}s"),
            _brief(status, 18),
            _brief(other, 36),
        )
    )
    title = (
        f"<strong>{_brief(record['id'])}</strong> · "
        f"{_brief(record['question'], 145)}"
    )
    header = "".join(
        f"<th style='padding:4px 7px;text-align:left'>"
        f"{html.escape(name)}</th>"
        for name in (
            "Expected → actual",
            "Tool",
            "Prior links",
            "Retrieval",
            "Evidence",
            "Tokens in/out",
            "Time",
            "Routing check",
            "Other checks",
        )
    )
    answer = record.get("answer") or ""
    summary = record.get("answer_summary") or ""
    if record["mode"] == "route-only":
        summary = "---"
    elif not summary and record.get("summary_source") == "unavailable":
        summary = "(summary unavailable)"
    elif not summary:
        summary = "(no final-answer summary)"
    one_line_answer = _brief(summary, 180)
    issues = "<div>" + "</div><div>".join(
        f"<b>{html.escape(kind)}:</b> "
        f"{_brief(', '.join(messages), 300)}"
        for kind, messages in checks.items() if messages
    ) + "</div>" if any(checks.values()) else "none"
    origin = f"{record.get('label', '?')} / {record.get('model', '?')}"
    trace_path = record.get("trace_path", "-")
    route = record.get("trace", {})
    model_calls = route.get("model_calls", [])
    generation_events = record.get("generation_events")
    tool_attempts = record.get("tool_internal_attempts")
    tool_text = (
        str(tool_attempts) if tool_attempts is not None
        else "not reported" if record.get("tool") else "---"
    )
    limit_hits = sum(
        call.get("finish_reason") == "length" for call in model_calls
    )
    prior_turns = record.get("previous", [])
    chosen = set(record.get("related_turn_ids", []))
    prior_html = "".join(
        f"<li>{'*' if item['turn'] in chosen else ' '} "
        f"#{item['turn']} {_brief(item['question'], 100)}: "
        f"{_brief(item['summary'], 120)} "
        f"({_brief(item.get('summary_source'), 15)})</li>"
        for item in prior_turns
    )
    refers_back = route.get("classification", {}).get(
        "refers_back", route.get("refers_back")
    )
    scope_note = ""
    if scanned is not None:
        scope_note = (
            f"<div><b>Scan:</b> {scanned} files scanned; "
            f"{record.get('files_with_mentions')} with matches; "
            f"{record.get('listed')} items listed; "
            f"{len(source_names)} source files in saved evidence.</div>"
        )
    elif record.get("passages_read") is not None:
        scope_note = (
            "<div><b>Passages actually read:</b> "
            f"{record['passages_read']} "
            f"(planned scope: {record.get('planned_passages')})."
            "</div>"
        )
    details = (
        f"<div><b>Scenario:</b> {_brief(origin)} · "
        f"<b>Turn:</b> {_brief(record.get('turn_number'))} · "
        f"<b>Trace:</b> {_brief(trace_path, 130)}</div>"
        f"<div><b>Checks:</b> {issues}</div>"
        f"<div><b>Summary provenance:</b> "
        f"{_brief(record.get('summary_source'))} · "
        f"<b>Trace/checkpoint agree:</b> "
        f"{_brief(record.get('summary_consistent'))}</div>"
        f"<div><b>Configured tool:</b> "
        f"{_brief(expected_tool(record.get('intent')) or '---')} · "
        f"<b>Evidence filenames:</b> {_brief(source_names, 350)}</div>"
        f"<div><b>Exact source paths:</b> "
        f"{_brief(record.get('source_paths', []), 800)}</div>"
        f"{scope_note}"
        f"<div><b>Shortcut:</b> {_brief(route.get('shortcut'))} · "
        f"<b>Fallback:</b> {_brief(route.get('fallback'))}</div>"
        f"<div><b>History IDs:</b> "
        f"{_brief(record.get('history_turn_ids', []))} · "
        f"<b>Refers back:</b> {_brief(refers_back)}</div>"
        f"<div><b>Generation-stage events:</b> "
        f"{generation_events if generation_events is not None else '---'}"
        f" · <b>Tool-internal attempts:</b> {_brief(tool_text)} · "
        f"<b>Length stops (generation stage):</b> {limit_hits} · "
        f"<b>Finish:</b> {_brief(record.get('finish_reason'))}</div>"
        "<details><summary>Earlier answer snapshots "
        "(* means linked by router)</summary><ul>"
        f"{prior_html}</ul></details>"
        "<div><b>Complete saved response:</b></div>"
        "<pre style='white-space:pre-wrap;max-height:300px;"
        "overflow:auto;font-size:12px'>"
        f"{html.escape(answer)}</pre>"
    )
    display(
        HTML(
            f"<div style='margin:8px 0 15px 0;font-size:12px'>"
            f"<div style='margin-bottom:5px'>{title}</div>"
            "<div style='overflow-x:auto'>"
            "<table style='border-collapse:collapse;width:100%;"
            "font-size:11px'><thead><tr>"
            f"{header}</tr></thead><tbody><tr>{body}</tr></tbody></table>"
            "</div>"
            f"<div style='margin:3px 0'><b>Recorded summary:</b> "
            f"{one_line_answer}</div>"
            f"<details style='margin-top:5px'><summary>Answer and routing "
            f"details</summary>{details}</details></div>"
        )
    )


def inspect_case(case_id, *, show_answer=True, show_prior=True):
    """Print the complete saved answer, related context, and trace data.

    Parameters
    ----------
    case_id : str
        ID shown by the compact diagnostic table.
    show_answer : bool, optional
        Whether to print the full answer, including any reasoning tags.
    show_prior : bool, optional
        Whether to print earlier linked questions and saved answer summaries.

    Returns
    -------
    dict
        Unmodified record collected by the evaluation harness.

    """
    record = CASE_INDEX[case_id]
    print(
        json.dumps(
            {
                key: value
                for key, value in record.items()
                if key not in ("answer", "trace")
            },
            indent=2,
            ensure_ascii=False,
        )
    )
    if show_prior:
        print("\nPREVIOUS TURN SNAPSHOT")
        for item in record.get("previous", []):
            marker = (
                "*"
                if item["turn"] in record.get("related_turn_ids", [])
                else " "
            )
            print(
                f"{marker} #{item['turn']}: {item['question']}\n"
                f"    {item['summary']}"
            )
    if show_answer:
        print("\nFULL ANSWER\n", record.get("answer", ""))
    print("\nNATIVE ROUTING DIAGNOSTICS")
    print(json.dumps(record.get("trace", {}), indent=2, ensure_ascii=False))
    return record


In [ ]:
def _previous_snapshot(context):
    """Capture KíLín's recorded prior-turn summaries for the router.

    Parameters
    ----------
    context : dict
        Active conversation state.

    Returns
    -------
    list of dict
        Prior-turn metadata with authoritative summary provenance.

    Notes
    -----
    Read ``conversation.turn.representation`` events first. When a trace
    is unavailable, use the same representation saved in turn parameters.
    Do not independently summarize the raw assistant response.

    """
    snapshot = []
    native_turns = _trace_turns(context["path"])
    for index, turn in enumerate(context["session"].turns, start=1):
        stages = native_turns.get(
            f"turn_{index:04d}", {}
        ).get("stages", {})
        traced = _trace_operation(
            stages, "01_question", "conversation_turn_representation"
        ).get("details", {})
        representation = _recorded_representation(turn, traced)
        snapshot.append(
            {
                "turn": index,
                "question": turn.question,
                "summary": representation["summary"],
                "summary_source": representation["origin"],
                "summary_consistent": representation["consistent"],
                "summary_sources": representation["sources"],
                "sources": _source_names(turn.evidence),
                "intent": turn.parameters.get("intent"),
                "tool": (turn.parameters.get("tool") or {}).get("name"),
            }
        )
    return snapshot


def _register(record):
    """Retain and display one result without changing any test outcome.

    Parameters
    ----------
    record : dict
        New unique case record.

    """
    if record["id"] in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {record['id']}")
    CASE_RESULTS.append(record)
    CASE_INDEX[record["id"]] = record
    show_case(record)


def run_case(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    retrieval=None,
    sources=(),
    forbid_sources=(),
    paths=(),
    forbid_paths=(),
    disallow=(),
    answer_terms=(),
    generated_function=None,
    choice="default",
    confirm="quick",
):
    """Run and record a full RAG turn with explicit context expectations.

    Parameters
    ----------
    context : dict
        Test conversation returned by ``conversation``.
    case_id : str
        Unique identifier used for later comparisons and follow-ups.
    question : str
        Unmodified user question.
    intent : str or None, optional
        Expected code intent; None records an exploratory observation.
    standalone : bool or None, optional
        Expected standalone classification; None omits this check.
    related_to : tuple of str, optional
        Earlier case IDs whose turns must be linked by the router.
    forbid_related : tuple of str, optional
        Earlier case IDs the router should not link to this question.
    retrieval : str or None, optional
        Expected retrieval mode: search, reuse, document, or none.
    sources : tuple of str, optional
        Source filenames that must appear in saved evidence.
    forbid_sources : tuple of str, optional
        Source filenames that must not occur in saved evidence.
    paths : tuple of str, optional
        Exact source-relative file paths expected in saved evidence.
    forbid_paths : tuple of str, optional
        Paths that must not be retrieved.
    disallow : tuple of str, optional
        Intents that must not be chosen, including ambiguous cases.
    answer_terms : tuple of str, optional
        Required final-answer keywords. A lightweight sanity check, not
        a semantic evaluation of code correctness.
    generated_function : str or None, optional
        Expected Python function name (parsed, never executed).
    choice : str or tuple, optional
        Scripted GUI document/list selection.
    confirm : str, optional
        GUI-equivalent confirmation mode, with network access disabled.

    Returns
    -------
    dict
        Recorded answer and machine-verifiable routing observations.

    Notes
    -----
    A routing mismatch is recorded without stopping the conversation.
    Later turns therefore test how the system handles imperfect history.
    Missing required previous turns is a test setup error and raises.

    """
    if intent is not None and intent not in code_intents():
        raise ValueError(f"Unknown intent: {intent}")
    expected_links = _expected_links(context, related_to)
    forbidden_links = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    before = len(session.turns)
    previous = _previous_snapshot(context)
    events = []
    choose, confirm_plan = _gui_callbacks(events, choice, confirm)
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "full",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": expected_links,
        "forbidden_related_ids": forbidden_links,
        "expected_retrieval": retrieval,
        "expected_sources": list(sources),
        "forbidden_sources": list(forbid_sources),
        "expected_paths": list(paths),
        "forbidden_paths": list(forbid_paths),
        "disallowed_intents": list(disallow),
        "expected_answer_terms": list(answer_terms),
        "expected_generated_function": generated_function,
        "previous": previous,
        "trace_path": str(context["path"].relative_to(project_dir)),
        "gui_events": events,
    }
    try:
        answer = session.ask(
            question,
            retrieval_count=RETRIEVAL_COUNT,
            choice_callback=choose,
            confirm_callback=confirm_plan,
        )
        if len(session.turns) != before + 1:
            raise RuntimeError("Expected exactly one committed turn.")
        turn = session.turns[-1]
        parameters = turn.parameters
        tool = parameters.get("tool") or {}
        native = _native_route(context["path"], len(session.turns))
        representation = _recorded_representation(
            turn, native.get("representation")
        )
        internal = tool.get("internal_generation") or {}
        attempts = internal.get("calls")
        record.update(
            {
                "answer": answer,
                "answer_summary": representation["summary"],
                "summary_source": representation["origin"],
                "summary_consistent": representation["consistent"],
                "summary_sources": representation["sources"],
                "turn_number": len(session.turns),
                "intent": parameters.get("intent"),
                "tool": tool.get("name"),
                "tool_status": tool.get("status"),
                "tool_summary": tool,
                "standalone": parameters.get("standalone"),
                "related_turn_ids": list(
                    parameters.get("related_turn_ids", [])
                ),
                "history_turn_ids": list(
                    parameters.get("history_turn_ids", [])
                ),
                "retrieval_mode": parameters.get("retrieval_mode"),
                "sources": _source_names(turn.evidence),
                "source_paths": _source_paths(turn.evidence),
                "evidence_count": len(turn.evidence),
                "files_scanned": tool.get("files_scanned"),
                "files_with_mentions": tool.get(
                    "files_with_mentions"
                ),
                "listed": tool.get("listed"),
                "passages_read": tool.get("passages_read"),
                "planned_passages": tool.get("passages"),
                "generation_events": (
                    len(native.get("model_calls", []))
                    if native.get("trace_available") else None
                ),
                "tool_internal_attempts": (
                    len(attempts) if isinstance(attempts, list)
                    else None
                ),
                "invalid_citations": parameters.get("invalid_citations", []),
                "completion": parameters.get("completion"),
                "usage": parameters.get("usage", {}),
                "finish_reason": parameters.get("finish_reason"),
                "trace": native,
            }
        )
        _grade_record(record)
        context["case_turn_ids"][case_id] = len(session.turns)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record


def probe_route(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    disallow=(),
):
    """Test routing against real conversation history without execution.

    Parameters
    ----------
    context : dict
        Active context with previously saved conversation turns.
    case_id : str
        Unique diagnostic case ID.
    question : str
        Question to route without saving or answering.
    intent : str or None, optional
        Intended router classification.
    standalone : bool or None, optional
        Expected independence from previous turns.
    related_to : tuple of str, optional
        Required previous case IDs.
    forbid_related : tuple of str, optional
        Previous case IDs that must not be linked.
    disallow : tuple of str, optional
        Intents that must not be selected for this request.

    Returns
    -------
    dict
        Routing-only decision and any diagnosed mismatches.

    Notes
    -----
    This uses KíLín's own ``route_question`` and the active session's
    private router accessor. It does not save an answer or execute a tool.

    """
    if intent is not None and intent not in code_intents():
        raise ValueError(f"Unknown intent: {intent}")
    required = _expected_links(context, related_to)
    forbidden = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "route-only",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": required,
        "forbidden_related_ids": forbidden,
        "disallowed_intents": list(disallow),
        "previous": _previous_snapshot(context),
        "trace_path": str(context["path"].relative_to(project_dir)),
    }
    try:
        config = project_type(session.project.profile.type_id)
        mark = session.usage_mark()
        decision = route_question(
            session._router_chat(), config, session.turns, question
        )
        record.update(
            {
                "intent": decision.intent,
                "tool": None,
                "standalone": decision.standalone,
                "related_turn_ids": list(decision.related_turn_ids),
                "retrieval_mode": config.intents[
                    decision.intent
                ].retrieval_mode,
                "sources": [],
                "answer": "(routing only: no answer)",
                "usage": session.usage_since(mark),
                "trace": {
                    "shortcut": decision.shortcut,
                    "fallback": decision.fallback,
                    "router_raw": decision.raw,
                    "dropped_turns": list(decision.dropped_turns),
                    "refers_back": decision.refers_back,
                },
            }
        )
        _grade_record(record)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record


In [ ]:
def show_results(*, prefix=None, group=None, mode=None):
    """Display a compact summary across selected recorded cases.

    Parameters
    ----------
    prefix : str or None, optional
        Limit case IDs to a prefix.
    group : str or None, optional
        Limit records to a named conversation group.
    mode : str or None, optional
        Limit to ``full`` or ``route-only`` cases.

    Returns
    -------
    list of dict
        Selected records.

    """
    rows = [
        record
        for record in CASE_RESULTS
        if (prefix is None or record["id"].startswith(prefix))
        and (group is None or record["label"] == group)
        and (mode is None or record["mode"] == mode)
    ]
    if not rows:
        print("No matching cases recorded.")
        return []
    headings = (
        "Case",
        "Model",
        "Expected",
        "Actual",
        "Links",
        "Route",
        "Scope",
        "Evidence files",
        "Warnings",
        "Seconds",
    )
    header = "".join(f"<th>{html.escape(name)}</th>" for name in headings)
    body = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 65)}</td>"
            for value in (
                item["id"],
                item.get("model", "-"),
                item.get("expected_intent") or "---",
                item.get("intent") or "-",
                str(item.get("related_turn_ids", [])),
                item["status"],
                item.get("scope_status", "---"),
                (
                    f"{len(item.get('sources', []))} retained / "
                    f"{item['files_scanned']} scanned"
                    if item.get("files_scanned") is not None
                    else str(len(item.get("sources", [])))
                    if item["mode"] == "full" else "---"
                ),
                ", ".join(
                    key for key in ("execution", "trace", "content")
                    if item.get("checks", {}).get(key)
                ) or "---",
                f"{item['seconds']:.2f}",
            )
        )
        + "</tr>"
        for item in rows
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody>"
            "</table></div>"
        )
    )
    return rows


def show_coverage():
    """Report coverage against all configured code intents.

    Returns
    -------
    dict
        Intent-to-observation mapping for the active experiment.

    """
    coverage = {}
    for intent, settings in code_intents().items():
        records = [
            item for item in CASE_RESULTS if item["expected_intent"] == intent
        ]
        statuses = [item["status"] for item in records]
        coverage[intent] = {
            "tool": (settings.get("tool") or {}).get("name"),
            "full": sum(item["mode"] == "full" for item in records),
            "probes": sum(item["mode"] == "route-only" for item in records),
            "matched": statuses.count("MATCH"),
            "review": statuses.count("REVIEW"),
            "errors": statuses.count("ERROR"),
        }
    heading = "".join(
        f"<th>{html.escape(name)}</th>"
        for name in (
            "Intent",
            "Expected tool",
            "Full",
            "Probes",
            "Match",
            "Review",
            "Errors",
        )
    )
    cells = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value)}</td>"
            for value in (
                intent,
                item["tool"] or "---",
                item["full"],
                item["probes"],
                item["matched"],
                item["review"],
                item["errors"],
            )
        )
        + "</tr>"
        for intent, item in coverage.items()
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{heading}</tr></thead><tbody>{cells}</tbody>"
            "</table></div>"
        )
    )
    return coverage


def show_answer_model_comparison():
    """Compare the same numbered prompts across answering models.

    Returns
    -------
    list of dict
        Matching full-pipeline answering-model records.

    """
    items = [item for item in CASE_RESULTS if item["id"].startswith("M_")]
    if not items:
        print("No answering-model comparison has been run.")
        return []
    items = sorted(
        items, key=lambda item: (item["id"].rsplit("_", 1)[-1], item["model"])
    )
    headers = (
        "Prompt",
        "Answer model",
        "Intent",
        "Sources",
        "Seconds",
        "Recorded summary",
        "Route",
    )
    head = "".join(f"<th>{html.escape(name)}</th>" for name in headers)
    rows = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 145)}</td>"
            for value in (
                item["id"].rsplit("_", 1)[-1],
                item["model"],
                item.get("intent"),
                ", ".join(item.get("sources", [])),
                f"{item['seconds']:.2f}",
                item.get("answer_summary") or "(summary unavailable)",
                item["status"],
            )
        )
        + "</tr>"
        for item in items
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:11px'>"
            "<table style='width:100%;border-collapse:collapse'>"
            f"<thead><tr>{head}</tr></thead><tbody>{rows}</tbody>"
            "</table></div>"
        )
    )
    return items


def show_diagnostic_totals():
    """Print routing outcomes and separate evidence/trace warnings.

    Returns
    -------
    dict
        Counts by execution mode and diagnostic category.

    """
    totals = {}
    for mode in ("full", "route-only"):
        items = [item for item in CASE_RESULTS if item["mode"] == mode]
        totals[mode] = {
            "cases": len(items),
            "match": sum(item["status"] == "MATCH" for item in items),
            "review": sum(item["status"] == "REVIEW" for item in items),
            "unscored": sum(item["status"] == "---" for item in items),
            "errors": sum(item["status"] == "ERROR" for item in items),
            "scope": sum(bool(item.get("checks", {}).get("scope"))
                         for item in items),
            "citation_warnings": sum(
                bool(item.get("invalid_citations")) for item in items
            ),
            "trace_warnings": sum(
                bool(item.get("checks", {}).get("trace"))
                for item in items
            ),
            "content_anchors": sum(
                bool(item.get("checks", {}).get("content"))
                for item in items
            ),
        }
    display(HTML(
        "<table style='font-size:12px;border-collapse:collapse'>"
        "<tr>" + "".join(f"<th>{name}</th>" for name in (
            "Mode", "Cases", "Match", "Review", "---", "Errors",
            "Scope flags", "Citation flags", "Trace flags",
            "Anchor flags"
        )) + "</tr>" + "".join(
            "<tr>" + "".join(f"<td>{_brief(value)}</td>" for value in (
                mode, item["cases"], item["match"], item["review"],
                item["unscored"], item["errors"], item["scope"],
                item["citation_warnings"], item["trace_warnings"],
                item["content_anchors"]
            )) + "</tr>" for mode, item in totals.items()
        ) + "</table>"
    ))
    return totals


def verify_saved_workspace():
    """Compare notebook full-turn records with committed trace checkpoints.

    Returns
    -------
    dict
        Number of verified answers and any field disagreements.

    Notes
    -----
    Routing probes have no saved turn and are deliberately excluded.

    """
    checked = 0
    problems = []
    cache = {}
    for record in CASE_RESULTS:
        if record["mode"] != "full" or record["status"] == "ERROR":
            continue
        path = project_dir / record["trace_path"]
        if path not in cache:
            try:
                with path.open(encoding="utf-8") as stream:
                    trace = json.load(stream)
                cache[path] = trace["sessions"]["05_conversations"][
                    "resume_state"
                ]["turns"]
            except (OSError, ValueError, KeyError, TypeError):
                cache[path] = []
        number = record.get("turn_number", 0)
        saved = (
            cache[path][number - 1]
            if 1 <= number <= len(cache[path]) else None
        )
        if saved is None:
            errors = ["missing committed turn"]
        else:
            params = saved.get("parameters", {})
            errors = []
            for field, observed in (
                ("answer", record.get("answer")),
                ("question", record.get("question")),
            ):
                if saved.get(field) != observed:
                    errors.append(field)
            for field in ("intent", "related_turn_ids", "history_turn_ids"):
                if params.get(field) != record.get(field):
                    errors.append(field)
            if len(saved.get("evidence", [])) != record.get(
                "evidence_count"
            ):
                errors.append("evidence count")
            if _source_paths(saved.get("evidence", [])) != (
                record.get("source_paths")
            ):
                errors.append("source paths")
            if _source_names(saved.get("evidence", [])) != record.get(
                "sources"
            ):
                errors.append("evidence sources")
            if (params.get("tool") or {}).get("name") != record.get(
                "tool"
            ):
                errors.append("tool")
            if params.get("retrieval_mode") != record.get(
                "retrieval_mode"
            ):
                errors.append("retrieval mode")
            if (params.get("representation") or {}).get(
                "summary"
            ) != record.get("answer_summary"):
                errors.append("recorded summary")
        record["workspace_mismatches"] = errors
        checked += 1
        if errors:
            problems.append({"case": record["id"], "fields": errors})
    print("Committed turns verified:", checked)
    print("Notebook/workspace discrepancies:", len(problems))
    for item in problems:
        print(f"  {item['case']}: {', '.join(item['fields'])}")
    return {"checked": checked, "mismatches": problems}


def annotate_case(case_id, *, quality=None, note=""):
    """Record a manual code-quality verdict without model grading.

    Parameters
    ----------
    case_id : str
        Existing case ID.
    quality : str or None, optional
        Human judgment such as ``correct``, ``mixed``, or ``incorrect``.
    note : str, optional
        Source lines or tests supporting the assessment.

    """
    CASE_INDEX[case_id]["manual_review"] = {
        "quality": quality,
        "note": note,
    }


CODE_REFERENCE_CASES = (
    ("D02", "kilin/rag.py", "RAGSession.ask"),
    ("D03", "kilin/conversations.py", "ConversationStore.rename"),
    ("A01", "kilin/answering.py", "route_question"),
    ("A03", "kilin/workspace.py", "prepare_sources_workspace"),
    ("H01", "kilin/gui/web/app.js", None),
    ("H02", "kilin/gui/web/index.html", None),
    ("H03", "kilin/project_types/code.json", None),
    ("H04", "tutorials/140_logging_trace.ipynb", None),
    ("X01", "kilin/rag.py", "RAGSession.ask"),
    ("X02", "kilin/conversations.py", "ConversationStore.create"),
)


def show_code_references():
    """Display independent definitions and per-case evidence coverage.

    Returns
    -------
    list of dict
        Reference-case records that have already executed.

    Notes
    -----
    AST definitions are computed from this experiment's copied files.
    Found words and source passages are insufficient to prove an
    explanation of control flow, side effects, or cross-file behavior.

    """
    rows = []
    observations = []
    for case_id, path, symbol in CODE_REFERENCE_CASES:
        item = CASE_INDEX.get(case_id)
        fact = (AST_FACTS.get((path, symbol)) if symbol else None)
        if item is not None:
            observations.append(item)
        source_ok = (
            "yes" if item and _has_source(
                item.get("source_paths", []), path
            ) else "no" if item else "not run"
        )
        ground = (
            f"{fact['kind']} at line {fact['line']}" if fact
            else "whole file" if symbol is None else "not in snapshot"
        )
        quality = (
            (item.get("manual_review") or {}).get("quality")
            if item else None
        ) or "pending"
        values = (case_id, path, symbol or "(file)", ground,
                  source_ok, quality)
        rows.append(
            "<tr>" + "".join(
                f"<td>{_brief(value, 105)}</td>" for value in values
            ) + "</tr>"
        )
    headers = (
        "Case", "Reference file", "Symbol", "Snapshot fact",
        "In evidence", "Manual quality",
    )
    head = "".join(f"<th>{html.escape(h)}</th>" for h in headers)
    display(HTML(
        "<div style='overflow-x:auto;font-size:12px'>"
        "<table style='border-collapse:collapse;width:100%'>"
        f"<thead><tr>{head}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table></div>"
    ))
    return observations


def export_evaluation():
    """Write local JSON/CSV records and the experiment manifest.

    Returns
    -------
    pathlib.Path
        Directory holding the saved evaluation files.

    """
    folder = project_dir / "outputs" / "notebook_evaluation"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "cases.json").write_text(
        json.dumps(CASE_RESULTS, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    with (folder / "cases.csv").open(
        "w", encoding="utf-8", newline=""
    ) as stream:
        columns = [
            "id",
            "mode",
            "label",
            "model",
            "router_model",
            "expected_intent",
            "intent",
            "expected_tool",
            "tool",
            "standalone",
            "related_turn_ids",
            "history_turn_ids",
            "retrieval_mode",
            "sources",
            "evidence_count",
            "source_paths",
            "expected_paths",
            "forbidden_paths",
            "expected_generated_function",
            "files_scanned",
            "files_with_mentions",
            "passages_read",
            "generation_events",
            "tool_internal_attempts",
            "status",
            "issues",
            "seconds",
            "trace_path",
            "answer_summary",
            "summary_source",
            "summary_consistent",
            "scope_status",
            "checks",
            "invalid_citations",
            "completion",
            "workspace_mismatches",
            "manual_review",
        ]
        assert len(columns) == len(set(columns))
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for record in CASE_RESULTS:
            writer.writerow(
                {
                    name: (
                        json.dumps(record.get(name), ensure_ascii=False)
                        if isinstance(record.get(name), (list, dict))
                        else record.get(name)
                    )
                    for name in columns
                }
            )
    manifest = {
        "experiment_id": EXPERIMENT_ID,
        "project": str(project_dir),
        "primary_model": MODEL_IDS[PRIMARY_MODEL],
        "router_model": ROUTER_MODEL,
        "comparison_models": [MODEL_IDS[k] for k in COMPARISON_MODELS],
        "allow_download": ALLOW_DOWNLOAD,
        "retrieval_count": RETRIEVAL_COUNT,
        "source_hashes": globals().get("SOURCE_HASHES", {}),
        "source_version": SOURCE_VERSION,
        "source_directories": ["kilin", "tutorials"],
        "extensions": list(CODE_EXTENSIONS),
        "code_intents": list(code_intents()),
        "summary_source": "conversation.turn.representation",
        "enabled_sections": {
            "long_tools": RUN_LONG_TOOLS,
            "file_review": RUN_FULL_SOURCE_READ,
            "code_review": RUN_CODE_REVIEW,
            "answer_models": RUN_MODEL_COMPARISON,
            "router_models": RUN_ROUTER_COMPARISON,
        },
        "conversations": CONVERSATION_INDEX,
        "ast_facts": list(globals().get("AST_FACTS", {}).values()),
        "ast_relations": globals().get("AST_RELATIONS", {}),
    }
    (folder / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print("Exported:", folder)
    return folder


### 2.1 Protocol preflight

The code project currently defines **14 intents**. If that vocabulary changes
in a development checkout, update the notebook's coverage contract rather
than silently treating an untested intent as covered. Tool names come from
the active project configuration, not hard-coded copies.


In [ ]:
PLANNED_INTENTS = {
    "small_talk", "code_explanation", "fact_finding", "code_generation",
    "debugging", "file_reading", "project_inventory", "mention_listing",
    "topic_listing", "mention_classification", "refine_results",
    "source_context", "conversation_followup", "rephrase_previous",
}
configured = set(code_intents())
assert PLANNED_INTENTS == configured, (
    "Code intent vocabulary changed. Revisit coverage. "
    f"Missing: {sorted(PLANNED_INTENTS - configured)}; "
    f"new: {sorted(configured - PLANNED_INTENTS)}"
)
print("Configured code intents:", len(configured))
show_coverage()


## 3. Create an initially empty code project

An empty project provides an essential negative control: it can answer
greetings and report that no source files have been imported, but cannot
legitimately cite KíLín code. Only the **experiment profile** gains the
seven selected extensions. KíLín's packaged type definition is unchanged.


In [ ]:
from kilin.workspace import prepare_sources_workspace

if project_dir.exists():
    raise FileExistsError(
        f"Experiment workspace already exists: {project_dir}"
    )
project = prepare_sources_workspace(
    [], "code", project_dir,
    settings={
        "extensions": list(CODE_EXTENSIONS),
        "allow_download": ALLOW_DOWNLOAD,
        "retrieval_count": RETRIEVAL_COUNT,
    },
)
assert set(project.profile.extensions) == set(CODE_EXTENSIONS)
assert list(project.source.iterdir()) == []
print("Created empty code project:", project.root)
print("Saved code extensions:", project.profile.extensions)


### 3.1 Empty-project behavior

A source-free answer is useful for detecting unsupported citations and
fabricated code details. These controls are deliberately run before any
project files are imported.


In [ ]:
with conversation("empty-project") as ctx:
    run_case(
        ctx, "E01", "hello", intent="small_talk",
        standalone=True, retrieval="none",
    )
    run_case(
        ctx, "E02", "Which source files are in this project?",
        intent="project_inventory", retrieval="none",
    )
    run_case(
        ctx, "E03", "Explain RAGSession.ask as implemented here.",
        intent="code_explanation", retrieval="search",
    )
    run_case(
        ctx, "E04", "Find all mentions of ConversationStore in this "
        "empty project.", intent="mention_listing",
    )
show_results(prefix="E")


## 4. Import the development snapshot and establish ground truth

Import only the `kilin/` package and the `tutorials/` notebooks into the
managed workspace. The normal scanner applies the saved extensions and
file-selection limits. The notebook records the hashes of **copied** files
and verifies them against the development originals, so later changes to
the checkout cannot silently alter this experiment.

AST reference facts are derived from those copied files, not from KíLín's
answers. They identify actual definitions and their line numbers; they do
not establish runtime behavior or semantic correctness. JavaScript, JSON,
and HTML are ordinary source files in this experiment.


In [ ]:
import ast
from hashlib import sha256

from kilin.workspace import add_project_sources

sources = [repo_root / "kilin", repo_root / "tutorials"]
if not all(path.is_dir() for path in sources):
    raise FileNotFoundError("Missing kilin/ or tutorials/ source folder.")
project = add_project_sources(project_dir, sources)

SOURCE_HASHES = {}
for subdir in ("kilin", "tutorials"):
    snapshot = project.source / subdir
    assert snapshot.is_dir(), f"Missing imported directory: {subdir}"
    for copied in sorted(snapshot.rglob("*")):
        if not copied.is_file():
            continue
        relative = copied.relative_to(project.source)
        original = repo_root / relative
        digest = sha256(copied.read_bytes()).hexdigest()
        SOURCE_HASHES[relative.as_posix()] = digest
        if not original.is_file():
            raise FileNotFoundError(f"Source disappeared: {original}")
        if sha256(original.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Source changed during copy: {relative}")

assert all(
    name.startswith(("kilin/", "tutorials/"))
    for name in SOURCE_HASHES
)
assert (project.source / "kilin/gui/web/app.js").is_file()
assert (project.source / "kilin/gui/web/index.html").is_file()
assert (project.source / "kilin/project_types/code.json").is_file()
assert (project.source / "tutorials/140_logging_trace.ipynb").is_file()
for extension in (".json", ".js", ".html"):
    assert any(name.endswith(extension) for name in SOURCE_HASHES)

print("Imported source files:", len(SOURCE_HASHES))
print("Python modules:", sum(n.endswith(".py") for n in SOURCE_HASHES))
print("Tutorials:", sum(n.startswith("tutorials/") for n in SOURCE_HASHES))
print("JavaScript/JSON/HTML:", sum(
    n.endswith((".js", ".json", ".html"))
    for n in SOURCE_HASHES
))
print("Source hashes recorded; originals remain unchanged.")


In [ ]:
def _definition(path, dotted_name):
    """Find a syntactic definition in a copied Python module.

    Parameters
    ----------
    path : str
        Source-relative Python file path.
    dotted_name : str
        Top-level class/function or a single class method.

    Returns
    -------
    dict or None
        File, symbol, kind, and first/last source line if present.

    """
    file = project.source / path
    if not file.is_file():
        return None
    tree = ast.parse(file.read_text(encoding="utf-8"))
    parts = dotted_name.split(".")
    body = tree.body
    node = None
    for part in parts:
        node = next(
            (item for item in body if isinstance(
                item, (ast.ClassDef, ast.FunctionDef,
                       ast.AsyncFunctionDef)
            ) and item.name == part),
            None,
        )
        if node is None:
            return None
        body = node.body
    return {
        "file": path,
        "symbol": dotted_name,
        "kind": type(node).__name__,
        "line": node.lineno,
        "end_line": node.end_lineno,
    }


REFERENCE_SYMBOLS = (
    ("kilin/rag.py", "RAGSession.ask"),
    ("kilin/rag.py", "RAGSession"),
    ("kilin/conversations.py", "ConversationStore.create"),
    ("kilin/conversations.py", "ConversationStore.rename"),
    ("kilin/answering.py", "route_question"),
    ("kilin/answering.py", "select_history"),
    ("kilin/workspace.py", "prepare_sources_workspace"),
    ("kilin/project.py", "configure_project"),
    ("kilin/project_type.py", "project_type"),
)
AST_FACTS = {
    (file, name): _definition(file, name)
    for file, name in REFERENCE_SYMBOLS
}
missing = [
    f"{path}:{symbol}" for (path, symbol), value in AST_FACTS.items()
    if value is None
]
if missing:
    print("Symbols absent from this development snapshot:", missing)
else:
    print("All", len(AST_FACTS), "reference symbols located.")
for fact in AST_FACTS.values():
    if fact:
        print(f"{fact['file']}:{fact['line']} {fact['symbol']}")


# Independent syntactic import and call relationships.
rag_tree = ast.parse(
    (project.source / "kilin/rag.py").read_text(encoding="utf-8")
)
imports = sorted({
    item.module for item in ast.walk(rag_tree)
    if isinstance(item, ast.ImportFrom) and item.module
    and item.module.startswith("kilin.")
})
rag_class = next(
    item for item in rag_tree.body
    if isinstance(item, ast.ClassDef) and item.name == "RAGSession"
)
ask_method = next(
    item for item in rag_class.body
    if isinstance(item, ast.FunctionDef) and item.name == "ask"
)
method_calls = sorted({
    item.func.id for item in ast.walk(ask_method)
    if isinstance(item, ast.Call)
    and isinstance(item.func, ast.Name)
})
AST_RELATIONS = {
    "file": "kilin/rag.py",
    "imports": imports,
    "RAGSession.ask_direct_calls": method_calls,
}
print("rag.py imports kilin.answering:", "kilin.answering" in imports)
print("RAGSession.ask calls select_history:",
      "select_history" in method_calls)


## 5. Direct routing and repository discovery

These standalone questions check inventory, named code definitions, the
retrieval of relevant implementation, import relationships, and exact
mentions. Paths such as `kilin/rag.py` are verified independently from the
answer text. A symbol name appearing in an answer is only a weak anchor.


In [ ]:
with conversation("direct-routes") as ctx:
    run_case(
        ctx, "D01", "Which source folders and file types are "
        "available in this code project?",
        intent="project_inventory", retrieval="none",
    )
    run_case(
        ctx, "D02", "Explain RAGSession.ask in kilin/rag.py "
        "and how it decides what evidence to retrieve.",
        intent="code_explanation", retrieval="search",
        paths=("kilin/rag.py",), answer_terms=("RAGSession",),
    )
    run_case(
        ctx, "D03", "How does ConversationStore.rename in "
        "kilin/conversations.py persist a conversation title?",
        intent="code_explanation", retrieval="search",
        paths=("kilin/conversations.py",),
        answer_terms=("rename",),
    )
    run_case(
        ctx, "D04", "Which imports from kilin.answering does "
        "kilin/rag.py use? Explain their role.",
        intent="code_explanation", retrieval="search",
        paths=("kilin/rag.py",),
    )
    run_case(
        ctx, "D05", "List every exact occurrence of RAGSession "
        "in kilin/rag.py.",
        intent="mention_listing", retrieval="none",
        choice="first20",
    )
show_results(prefix="D")


## 6. Code referents and multi-turn continuity

These are real conversations with committed answers. The notebook checks
that source-specific references survive follow-up questions and explicit
switches between implementation files. Router related-turn IDs and the
actual history selected for generation are reported separately.


### 6.1 Source-file and symbol switches

An explicit module/function name establishes a preferred source scope. An
anaphoric follow-up should retain that source until another file or
symbol is introduced. We test source identity independently of intent.


In [ ]:
with conversation("code-relations") as ctx:
    run_case(
        ctx, "A01", "Explain route_question in kilin/answering.py.",
        intent="code_explanation", standalone=True,
        paths=("kilin/answering.py",),
    )
    run_case(
        ctx, "A02", "How does that function validate the "
        "router's selected related turns?",
        intent="code_explanation", standalone=False,
        related_to=("A01",), paths=("kilin/answering.py",),
    )
    run_case(
        ctx, "A03", "Now explain prepare_sources_workspace "
        "in kilin/workspace.py instead.",
        intent="code_explanation", paths=("kilin/workspace.py",),
    )
    run_case(
        ctx, "A04", "Does this second function modify the "
        "original source directory or make a copy?",
        intent="code_explanation", standalone=False,
        related_to=("A03",), paths=("kilin/workspace.py",),
    )
    probe_route(
        ctx, "A05", "Return to route_question: how does it "
        "handle malformed router output?",
        intent="code_explanation", standalone=False,
        related_to=("A01",),
    )
    probe_route(
        ctx, "A06", "Compare those two functions' roles in "
        "the creation and answering pipelines.",
        intent="code_explanation", standalone=False,
    )
show_results(prefix="A")


### 6.2 Previously answered questions, follow-ups, and rephrasing

A request to rephrase the last answer is not a new code search. A question
about the last answer can often reuse its evidence. A new named symbol
should instead trigger the relevant code retrieval policy.


In [ ]:
with conversation("code-continuity") as ctx:
    run_case(
        ctx, "C01", "Explain the role of ConversationStore in "
        "kilin/conversations.py.",
        intent="code_explanation", paths=("kilin/conversations.py",),
    )
    run_case(
        ctx, "C02", "What does its create method save?",
        intent="code_explanation", standalone=False,
        related_to=("C01",), paths=("kilin/conversations.py",),
    )
    run_case(
        ctx, "C03", "According to your previous answer, "
        "what information is recorded for the new conversation?",
        intent="conversation_followup", standalone=False,
        related_to=("C02",), retrieval="reuse",
    )
    run_case(
        ctx, "C04", "Rewrite that answer as two short bullets "
        "without adding any new facts.",
        intent="rephrase_previous", standalone=False,
        related_to=("C03",), retrieval="reuse",
    )
    probe_route(
        ctx, "C05", "Now explain configure_project in "
        "kilin/project.py.", intent="code_explanation",
    )
    probe_route(
        ctx, "C06", "What exactly did you say in the second bullet?",
        intent="conversation_followup", standalone=False,
        related_to=("C04",),
    )
show_results(prefix="C")


## 7. Previous-result tools versus new searches

Exact mention listing, filtering the displayed subset, and showing the
source context of a cited occurrence are different operations. Neither
filtering nor source-context inspection should silently start an
unrestricted search. Routing-only probes examine the boundaries without
running expensive classification and semantic scanning.


In [ ]:
with conversation("results-tools") as ctx:
    run_case(
        ctx, "S01", "List every exact occurrence of RAGSession "
        "in kilin/rag.py.",
        intent="mention_listing", retrieval="none",
        choice="first20",
    )
    run_case(
        ctx, "S02", "Show the surrounding source text of the "
        "first occurrence you just listed.",
        intent="source_context", standalone=False,
        related_to=("S01",), retrieval="none",
    )
    run_case(
        ctx, "S03", "Of those listed occurrences, keep only "
        "the ones in class or function definitions.",
        intent="refine_results", standalone=False,
        related_to=("S01",), retrieval="none",
    )
    probe_route(
        ctx, "S04", "Find new passages about how history "
        "selection works; do not just reuse that list.",
        intent="topic_listing", disallow=("source_context",),
    )
    probe_route(
        ctx, "S05", "Classify all exact uses of RAGSession "
        "as definitions, imports, or other references.",
        intent="mention_classification",
    )
    probe_route(
        ctx, "S06", "Filter only those earlier results for "
        "calls that construct a new RAGSession.",
        intent="refine_results", standalone=False,
        related_to=("S01",),
    )
show_results(prefix="S")


### 7.1 Display, export and cancellation choices

These cases exercise the same noninteractive GUI-equivalent callbacks as
the same noninteractive evaluation interface. Saving an exhaustive list is opt-in. A cancelled
expensive operation should not be mistaken for a successful exhaustive
answer. All network confirmation requests are always cancelled.


In [ ]:
with conversation("gui-paths") as ctx:
    run_case(
        ctx, "G01", "Which files are in the code project?",
        intent="project_inventory", choice="first20",
    )
    run_case(
        ctx, "G02", "List every exact mention of RAGSession "
        "throughout the code repository.",
        intent="mention_listing", choice="first20",
    )
    run_case(
        ctx, "G03", "Find every comment that discusses "
        "routing ambiguity rather than matching literal words.",
        intent="topic_listing", confirm="cancel",
    )
    if RUN_GUI_EXPORT_TEST:
        run_case(
            ctx, "G04", "List every exact mention of RAGSession "
            "throughout the code repository.",
            intent="mention_listing", choice="save_all",
        )
show_results(prefix="G")


## 8. Routing boundaries (lightweight probes)

The following cases cover **all 14 code-project intents**, with explicit
negative contrasts between exact mentions, semantic passages, factual
extraction, whole-file reading, code explanation and code generation.
A probe invokes only the router; it never executes the selected tool.
Where multiple routes are defensible, use an exploratory unscored case
instead of declaring an arbitrary route correct.


In [ ]:
with conversation("routing-boundaries") as ctx:
    run_case(
        ctx, "B00", "Explain RAGSession.ask in kilin/rag.py.",
        intent="code_explanation", paths=("kilin/rag.py",),
    )
    run_case(
        ctx, "B00b", "List exact occurrences of RAGSession "
        "in kilin/rag.py.", intent="mention_listing",
        choice="first20",
    )
    probe_route(ctx, "B01", "hello", intent="small_talk")
    probe_route(
        ctx, "B02", "Explain how RAGSession.ask chooses evidence.",
        intent="code_explanation",
    )
    probe_route(
        ctx, "B03", "What value does code.json specify for "
        "default_intent?", intent="fact_finding",
    )
    probe_route(
        ctx, "B04", "Write a new Python helper that returns "
        "the conversation names from a ConversationStore.",
        intent="code_generation",
    )
    probe_route(
        ctx, "B05", "Debug why a requested source filename "
        "might select a generated report with a similar name.",
        intent="debugging",
    )
    probe_route(
        ctx, "B06", "Read the entire kilin/gui/_paths.py file "
        "and review its implementation.", intent="file_reading",
    )
    probe_route(
        ctx, "B07", "How many notebooks are included in the "
        "project?", intent="project_inventory",
    )
    probe_route(
        ctx, "B08", "Find all exact uses of RAGSession.",
        intent="mention_listing",
    )
    probe_route(
        ctx, "B09", "Which files discuss conversation "
        "persistence by meaning?", intent="topic_listing",
    )
    probe_route(
        ctx, "B10", "Classify all exact occurrences of "
        "RAGSession as definitions or uses.",
        intent="mention_classification",
    )
    probe_route(
        ctx, "B11", "Of those listed instances, keep only "
        "the uses inside functions.",
        intent="refine_results", standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx, "B12", "Show the source context around the "
        "first occurrence you just listed.",
        intent="source_context", standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx, "B13", "What was your main point about "
        "RAGSession.ask?", intent="conversation_followup",
        standalone=False, related_to=("B00",),
    )
    probe_route(
        ctx, "B14", "Rewrite your explanation of RAGSession.ask "
        "in two sentences.", intent="rephrase_previous",
        standalone=False,
    )
    probe_route(
        ctx, "B15", "Which functions import code from "
        "kilin.answering?", intent="fact_finding",
    )
    probe_route(
        ctx, "B16", "Find the literal text 'code_graph' "
        "throughout the repository.", intent="mention_listing",
    )
    probe_route(
        ctx, "B17", "Find comments about source filtering "
        "without requiring an exact keyword.",
        intent="topic_listing",
    )
    probe_route(
        ctx, "B18", "Is kilin/project_types/code.json a "
        "different type of source from kilin/rag.py?",
        intent=None,
    )
    probe_route(
        ctx, "B19", "Explain why source hooks use a "
        "fallback for unrecognized file extensions.",
        intent="code_explanation",
    )
    probe_route(
        ctx, "B20", "Search kilin/project_types/code.json "
        "for the actual default intent setting.",
        intent="file_reading",
    )
    probe_route(
        ctx, "B21", "Add a new parser for JavaScript files.",
        intent="code_generation",
    )
    probe_route(
        ctx, "B22", "Which files handle generating a PDF report?",
        intent="topic_listing",
    )
    probe_route(
        ctx, "B23", "Tell me where KíLín makes the decision "
        "to ignore source files above the size limit.",
        intent="code_explanation",
    )
    probe_route(
        ctx, "B24", "Give me a full list of saved project "
        "profile settings.", intent="fact_finding",
    )
show_results(prefix="B")


## 9. Cross-format scope and source attribution

The development repository includes Python, JSON, JavaScript, HTML and
Jupyter notebooks. Each format is indexed through KíLín's normal text and
source extraction. The purpose here is **not** to compare parsers, but to
verify that answers use the specific files requested, and do not substitute
a similarly named output or an unrelated source.


In [ ]:
with conversation("hard-scope") as ctx:
    run_case(
        ctx, "H01", "Explain how renderProjectTypes in "
        "kilin/gui/web/app.js builds the project-type choices.",
        intent="code_explanation", paths=("kilin/gui/web/app.js",),
    )
    run_case(
        ctx, "H02", "How does kilin/gui/web/index.html load "
        "the JavaScript application code?",
        intent="code_explanation",
        paths=("kilin/gui/web/index.html",),
    )
    run_case(
        ctx, "H03", "Explain the default intent and "
        "retrieval policy in kilin/project_types/code.json.",
        intent="code_explanation",
        paths=("kilin/project_types/code.json",),
    )
    run_case(
        ctx, "H04", "What does tutorial "
        "tutorials/140_logging_trace.ipynb demonstrate?",
        intent="code_explanation",
        paths=("tutorials/140_logging_trace.ipynb",),
    )
    probe_route(
        ctx, "H05", "Which source stores GUI default model choices?",
        intent="fact_finding",
    )
    probe_route(
        ctx, "H06", "Explain how the Python engine coordinates "
        "with the JavaScript frontend.", intent="code_explanation",
    )
    probe_route(
        ctx, "H07", "Show me the exact lines from "
        "index.html that include app.js.", intent="file_reading",
    )
show_results(prefix="H")


### 9.1 Definition and code-graph ground-truth checks

These questions target definitions that were checked directly in the copied
snapshot using Python's AST. They also test multi-module source coverage.
A retrieved filename and a correct symbol name do not guarantee a correct
explanation of callers or control flow; assess those claims manually.


In [ ]:
with conversation("code-ground-truth") as ctx:
    run_case(
        ctx, "X01", "Where is RAGSession.ask defined? Explain "
        "its important arguments and return value.",
        intent="code_explanation", paths=("kilin/rag.py",),
        answer_terms=("ask",),
    )
    run_case(
        ctx, "X02", "How does ConversationStore.create "
        "manage conversation identity and optional branching?",
        intent="code_explanation",
        paths=("kilin/conversations.py",),
    )
    run_case(
        ctx, "X03", "Explain how kilin/rag.py uses "
        "kilin/answering.py to route questions. Cite both files.",
        intent="code_explanation", paths=(
            "kilin/rag.py", "kilin/answering.py",
        ),
    )
    run_case(
        ctx, "X04", "Explain how project_type in "
        "kilin/project_type.py loads the configured project type.",
        intent="code_explanation",
        paths=("kilin/project_type.py",),
    )
    run_case(
        ctx, "X05", "What role does code_graph play in "
        "the code project's capabilities configuration?",
        intent="code_explanation",
        paths=("kilin/project_types/code.json",),
    )
show_results(prefix="X")


## 10. Expensive end-to-end tools (opt-in)

Routing probes cover all 14 intents cheaply. The optional cases here
execute the expensive tools and distinguish a **Quick sample** from a
**Continue/full** scan. Do not interpret a Quick result as exhaustive.
Full semantic and fact-extraction scans may make many local model calls.


In [ ]:
if RUN_LONG_TOOLS:
    with conversation("optional-review") as ctx:
        run_case(
            ctx, "L01", "Which files describe how the system "
            "reuses previous-turn evidence, by meaning rather "
            "than by exact word matching?",
            intent="topic_listing", confirm="continue",
        )
        run_case(
            ctx, "L02", "Give a complete list of profile setting "
            "names supported by kilin/project.py.",
            intent="fact_finding", confirm="continue",
        )
        run_case(
            ctx, "L03", "Classify all exact occurrences of "
            "RAGSession as definitions, imports, or uses.",
            intent="mention_classification", confirm="continue",
        )
    show_results(prefix="L")
else:
    print("Full model-backed scans omitted: RUN_LONG_TOOLS=False")

if RUN_FULL_SOURCE_READ:
    with conversation("optional-file-review") as ctx:
        run_case(
            ctx, "F01", "Read all of kilin/gui/_paths.py and "
            "review its interface and behavior.",
            intent="file_reading", confirm="continue",
        )
    show_results(prefix="F")
else:
    print("Whole-file review omitted: RUN_FULL_SOURCE_READ=False")


## 11. Debugging and code generation

These tests ask KíLín to inspect existing code and propose changes **as
text only**. The notebook does not apply generated patches, execute model-
written programs, change source files, or invoke an unsafe interpreter.
For correctness, inspect the cited functions and verify generated code
independently before using it in a real repository.


In [ ]:
with conversation("code-generation") as ctx:
    run_case(
        ctx, "J01", "Investigate how a named source file is "
        "resolved by kilin/tools/_shared/choose_document.py. "
        "What could make a similarly named generated output "
        "appear instead of the requested source? Give checks "
        "to confirm the behavior before suggesting a fix.",
        intent="debugging",
    )
    run_case(
        ctx, "J02", "Write a small standalone Python function "
        "list_conversation_names(store) that reads the names "
        "from ConversationStore.conversations(). Follow "
        "KíLín's documentation style and 79-character limit. "
        "Return the function as text, not a patch.",
        intent="code_generation", answer_terms=("def ",),
        generated_function="list_conversation_names",
    )
    probe_route(
        ctx, "J03", "Modify the JSON project type configuration "
        "to enable source suffixes .js and .html.",
        intent="code_generation",
    )
    if RUN_CODE_REVIEW:
        run_case(
            ctx, "J04", "Review the logic that rejects source "
            "files exceeding a profile size limit. Identify "
            "a concrete bug if supported, otherwise explain "
            "how to test it; do not invent a bug.",
            intent="debugging",
        )
show_results(prefix="J")


## 12. Controlled answering-model comparison (opt-in)

Both models receive identical source snapshots and independent
conversations. The first turn is a direct, file-specific code explanation,
not an expensive report that might fail before creating an antecedent.
A follow-up runs **only when** the first answer was committed and includes
the named source; otherwise the comparison reports the missing seed instead
of interpreting an ungrounded pronoun as a model-quality result.


In [ ]:
if RUN_MODEL_COMPARISON:
    for model_key in dict.fromkeys(COMPARISON_MODELS):
        group = f"answer-model-{model_key}"
        with conversation(group, model=model_key) as ctx:
            seed = f"M_{model_key}_01"
            first = run_case(
                ctx, seed,
                "Explain RAGSession.ask in kilin/rag.py and "
                "describe how it retrieves evidence.",
                intent="code_explanation", standalone=True,
                paths=("kilin/rag.py",),
            )
            valid = (
                first["status"] != "ERROR"
                and not first.get("completion")
                and _has_source(
                    first.get("source_paths", []), "kilin/rag.py"
                )
            )
            if not valid:
                print("Skipping ungrounded model follow-ups:", model_key)
                continue
            second = f"M_{model_key}_02"
            run_case(
                ctx, second,
                "Which parameters does that method accept, and "
                "what do they control?",
                intent="code_explanation", standalone=False,
                related_to=(seed,), paths=("kilin/rag.py",),
            )
            run_case(
                ctx, f"M_{model_key}_03",
                "Compare that method with "
                "ConversationStore.create in kilin/conversations.py.",
                intent="code_explanation", standalone=False,
                paths=("kilin/rag.py", "kilin/conversations.py"),
            )
    show_answer_model_comparison()
else:
    print("Answering-model comparison omitted.")


## 13. Controlled router-model comparison (opt-in)

The router comparison changes the router only; the answering model and
source snapshot stay fixed. Branches inherit the same committed seed turn.
If the seed fails to establish the source, no branch-based continuation is
used. Probes test decisions without running their selected tools.


In [ ]:
if RUN_ROUTER_COMPARISON:
    with conversation("router-seed") as seed_ctx:
        first = run_case(
            seed_ctx, "Rseed00",
            "Explain ConversationStore in kilin/conversations.py.",
            intent="code_explanation",
            paths=("kilin/conversations.py",),
        )
        parent_id = seed_ctx["info"].conversation_id
        has_seed = (
            first["status"] != "ERROR"
            and not first.get("completion")
            and _has_source(
                first.get("source_paths", []),
                "kilin/conversations.py",
            )
        )
    if not has_seed:
        print("Router comparison skipped: no grounded seed turn.")
    else:
        for index, router in enumerate(COMPARISON_ROUTERS, 1):
            group = f"router-model-{index}"
            with conversation(
                group, router=router, branch_from=parent_id
            ) as ctx:
                if len(ctx["session"].turns) != 1:
                    raise RuntimeError("Router seed not restored.")
                ctx["case_turn_ids"]["Rseed00"] = 1
                probe_route(
                    ctx, f"R{index}_01",
                    "What does its rename method persist?",
                    intent="code_explanation", standalone=False,
                    related_to=("Rseed00",),
                )
                probe_route(
                    ctx, f"R{index}_02",
                    "List exact mentions of RAGSession.",
                    intent="mention_listing", standalone=True,
                )
                probe_route(
                    ctx, f"R{index}_03",
                    "Find files about GUI and backend communication.",
                    intent="topic_listing",
                )
                probe_route(
                    ctx, f"R{index}_04",
                    "How does that class handle creating branches?",
                    intent="code_explanation", standalone=False,
                    related_to=("Rseed00",),
                )
        show_results(prefix="R")
else:
    print("Router-model comparison omitted.")


## 14. Review and export

Review routing, evidence scope, execution, content anchors and trace
consistency **independently**. The saved checkpoint is reconciled against
the notebook's full answer, chosen intent, history, source paths and KíLín
recorded summary. Probes do not have committed answers to reconcile.

The reference table shows AST-derived definition locations from the copied
snapshot. It is intentionally not a model-generated evaluation score.


In [ ]:
reconciliation = verify_saved_workspace()
coverage = show_coverage()
diagnostics = show_diagnostic_totals()
code_references = show_code_references()
all_end_to_end = show_results(mode="full")
all_routing = show_results(mode="route-only")
print("Recorded cases:", len(CASE_RESULTS))
print("Conversations:", len(CONVERSATION_INDEX))


### 14.1 Inspect failures and annotate code quality

A routing `MATCH` is a routing-only claim. A correct filename can still
support an incorrect implementation description, and valid syntax may
still be unsuitable code. Review source-grounded cases individually and
record a qualitative verdict with `annotate_case`, for example:

```python
annotate_case(
    "D02", quality="mixed",
    note="Retrieved rag.py, but misidentified the history-selection step.",
)
```

False-positive citation warnings can occur when source text contains
literal numbered references. Verify attribution before classifying a
citation as fabricated.


In [ ]:
needs_review = [
    item for item in CASE_RESULTS
    if item["status"] in ("REVIEW", "ERROR")
]
for item in needs_review:
    print(
        f"{item['id']:<17} {item['status']:<7} "
        f"{', '.join(item.get('issues', []))}"
    )
print("Routing reviews and errors:", len(needs_review))
other = [
    item for item in CASE_RESULTS
    if any(
        item.get("checks", {}).get(name)
        for name in ("scope", "execution", "trace", "content")
    )
]
print("Separate scope/evidence/trace flags:", len(other))


### 14.2 Save the evaluation

Exports are stored **inside this experiment workspace**:

- `cases.json`: complete raw answers, recorded summaries, routing decisions,
  source paths, tools, usage and trace diagnostics;
- `cases.csv`: compact comparisons, distinct scan/evidence/call counters;
- `manifest.json`: model settings, Git version/status, imported file hashes,
  expected intents, AST definitions, conversation names and enabled flags.

Source code, notebook outputs and model reasoning may be present in the
workspace and should be treated as private development material.


In [ ]:
evaluation_dir = export_evaluation()


## 15. Interpreting the results

Diagnose discrepancies by layer before changing KíLín:

1. **Routing:** wrong intent, wrong tool or wrong preceding-turn link.
2. **Retrieval scope:** named module missing, stale symbol resolution,
   unrelated evidence, or insufficient cross-file coverage.
3. **Code facts:** compare against AST definitions, source lines, imports
   and actual implementation—not against plausible descriptions.
4. **Tool execution:** distinguish Quick samples, Continue/full scans,
   cancellation, incomplete generation and tool-internal calls.
5. **Conversational fidelity:** confirm that a prior answer succeeded
   before interpreting pronouns and model comparisons.
6. **Trace integrity:** compare original saved answers and KíLín-authored
   summaries against committed conversation state.

The benchmark follows the **development checkout**, not a frozen release.
The snapshot hashes and recorded Git state make every run auditable.
Change one KíLín component at a time and use the same scenarios to verify
whether a particular failure improves without weakening existing behavior.
